# FloodValue — обучение основного метода в Colab

Обёртка над **той же командой**, что локально: `python -m floodvalue train` (D-009).
Логики обучения здесь нет — только окружение, данные, запуск и выгрузка весов.

Шаги: параметры → клон репозитория → uv и окружение → Sen1Floods11 (`python -m floodvalue data`)
→ обучение → sha256 и выгрузка весов (Google Drive или скачивание).

- **Среда выполнения**: *Runtime → Change runtime type → GPU*. Без GPU ячейки выполнятся
  на CPU (полное обучение на CPU займёт очень долго — для проверки проводки есть `MODE = "smoke"`).
- **GPU-torch** ставится в отдельный venv вне репозитория `scripts/setup_gpu_env.sh`
  (D-024, индекс `cu128`); `uv.lock` (CPU-torch для инференса у жюри, D-003) не меняется.
- **Данные** Sen1Floods11 качаются из официального публичного бакета `gs://sen1floods11`
  (v1.1, `docs/data.md`) в `FLOODVALUE_DATA_DIR`, вне git. Test-метки запечатаны кодом
  (`SealedTestError`): обучение читает только train/dev/validation замороженного split.
- **Приватный репозиторий**: добавьте секрет Colab `GITHUB_TOKEN` (значок ключа слева) —
  токен используется только для `git clone` и сразу удаляется из `remote`.

Каждый параметр ниже можно переопределить переменной окружения с тем же именем
(`FLOODVALUE_NB_<ИМЯ>`), так ноутбук прогоняется и вне Colab.

In [ ]:
# --- Параметры -------------------------------------------------------------------------
import os

def param(name, default):
    return os.environ.get(f"FLOODVALUE_NB_{name}", default)

REPO_URL = param("REPO_URL", "https://github.com/Chichetin/kosmo-moskva_umstvenno_ustalye_34.git")
REF = param("REF", "main")                    # ветка, тег или commit
WORKDIR = param("WORKDIR", "/content")        # куда клонировать и где хранить данные/результаты
MODE = param("MODE", "full")                  # "full" — python -m floodvalue train (5 сидов x 60 эпох -> models/unet/)
                                              # "smoke" — 1 эпоха на 4 чипах: проверка проводки, веса не для сдачи
USE_GPU = param("USE_GPU", "auto")            # "auto" | "1" | "0"
DATA_DIR = param("DATA_DIR", "")              # пусто -> <WORKDIR>/data; можно путь на Drive, чтобы не качать заново
SAVE_TO_DRIVE = param("SAVE_TO_DRIVE", "1")   # "1" — копия архива весов в MyDrive/floodvalue (в Colab)

REPO_DIR = os.path.join(WORKDIR, "floodvalue")
DATA_DIR = DATA_DIR or os.path.join(WORKDIR, "data")
OUT_DIR = os.path.join(WORKDIR, "outputs")
GPU_VENV = os.path.join(WORKDIR, "floodvalue-gpu")   # свой venv, не общий $HOME/.cache (D-024)
assert MODE in ("full", "smoke"), MODE
print(f"REPO_URL={REPO_URL} REF={REF} MODE={MODE}\nREPO_DIR={REPO_DIR}\nDATA_DIR={DATA_DIR}\nOUT_DIR={OUT_DIR}")

In [ ]:
# --- Вспомогательное: запуск команд с потоковым выводом и проверкой кода возврата ---------
import shlex, shutil, subprocess, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def sh(cmd, cwd=None, env=None, secret=None):
    """Run a shell command, stream its output, fail the cell on a non-zero exit code."""
    shown = cmd.replace(secret, "***") if secret else cmd
    print(f"$ {shown}", flush=True)
    proc = subprocess.Popen(["bash", "-c", f"set -eo pipefail; {cmd}"], cwd=cwd,
                            env={**os.environ, **(env or {})}, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line.replace(secret, "***") if secret else line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"exit code {proc.returncode}: {shown}")

print("Colab" if IN_COLAB else "не Colab (локальный прогон)")

In [ ]:
if IN_COLAB and any(p.startswith("/content/drive") for p in (DATA_DIR, WORKDIR)):
    from google.colab import drive
    drive.mount("/content/drive")

## 1. Клон репозитория

In [ ]:
token = ""
if IN_COLAB:
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:  # no secret defined / no access granted -> public clone
        token = ""
clone_url = REPO_URL
if token and REPO_URL.startswith("https://"):
    clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@", 1)

os.makedirs(WORKDIR, exist_ok=True)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # Weights in models/ are Git LFS objects; training does not need them.
    sh(f"git clone {shlex.quote(clone_url)} {shlex.quote(REPO_DIR)}",
       env={"GIT_LFS_SKIP_SMUDGE": "1"}, secret=token or None)
    sh(f"git remote set-url origin {shlex.quote(REPO_URL)}", cwd=REPO_DIR)
# Always the freshly fetched REF (branch, tag or commit), never a stale local branch.
sh(f"git fetch --quiet {shlex.quote(clone_url)} {shlex.quote(REF)}", cwd=REPO_DIR,
   env={"GIT_LFS_SKIP_SMUDGE": "1"}, secret=token or None)
sh("git checkout --quiet --detach FETCH_HEAD", cwd=REPO_DIR, env={"GIT_LFS_SKIP_SMUDGE": "1"})
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("commit", COMMIT)

## 2. uv и окружение

`uv sync --frozen` — окружение строго по `uv.lock` (CPU-torch, как у жюри). При наличии GPU
поверх ставится отдельный venv с GPU-torch: `scripts/setup_gpu_env.sh` (D-024), обучение
идёт его python. Команда обучения одна и та же.

In [ ]:
if shutil.which("uv") is None:
    sh(f"{shlex.quote(sys.executable)} -m pip install --quiet uv")
UV = shutil.which("uv") or os.path.join(os.path.dirname(sys.executable), "uv")
sh(f"{UV} --version")
sh(f"{UV} sync --frozen", cwd=REPO_DIR)

has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(
    ["nvidia-smi"], capture_output=True).returncode == 0
assert USE_GPU in ("auto", "1", "0"), USE_GPU
gpu = {"auto": has_gpu, "1": True, "0": False}[USE_GPU]
if gpu:
    sh("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv")
    sh("bash scripts/setup_gpu_env.sh", cwd=REPO_DIR,
       env={"FLOODVALUE_GPU_VENV": GPU_VENV, "PATH": os.path.dirname(UV) + ":" + os.environ["PATH"]})
    PY = f"{shlex.quote(os.path.join(GPU_VENV, 'bin', 'python'))}"
else:
    PY = f"{UV} run --frozen python"
ENV = {"FLOODVALUE_DATA_DIR": DATA_DIR, "FLOODVALUE_OUTPUT_DIR": OUT_DIR}
sh(f"{PY} -c 'import torch; print(\"torch\", torch.__version__, \"cuda\", torch.cuda.is_available())'",
   cwd=REPO_DIR, env=ENV)

## 3. Sen1Floods11

`python -m floodvalue data` (`make data`): качает недостающие HandLabeled-слои S1Hand, LabelHand,
JRCWaterHand и сплиты из `gs://sen1floods11` v1.1 (≈ 0,7 ГБ, без авторизации), проверяет md5,
делает QC и пишет `interim/qc.csv`, `interim/source_manifest_s1.json`. Повторный запуск ничего не
качает. Чтобы не качать в каждой сессии, укажите `DATA_DIR` на Google Drive — ячейка перед разделом 1 смонтирует
Drive, если `DATA_DIR` или `WORKDIR` лежат в `/content/drive`.

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
sh(f"{PY} -m floodvalue data", cwd=REPO_DIR, env=ENV)

## 4. Обучение

- `full`: `python -m floodvalue train` — ансамбль из 5 сидов конфигурации по умолчанию, члены
  в `<OUT_DIR>/unet-runs/`, собранный ансамбль — в `models/unet/` клона.
- `smoke`: `python -m floodvalue.model.unet.train --epochs 1 --max-chips 4 ...` — тот же модуль
  обучения с его штатными флагами, одна модель; проверяет, что окружение и данные доходят до
  обучения. Эти веса не для сдачи.
- Возобновления нет: при обрыве сессии Colab прогоны членов в `/content` пропадут — для
  full-режима задайте `WORKDIR` на Drive (`/content/drive/MyDrive/…`).

In [ ]:
import time
TRAIN_STARTED = time.time()
if MODE == "full":
    sh(f"{PY} -m floodvalue train --out {shlex.quote(OUT_DIR)}", cwd=REPO_DIR, env=ENV)
    ARTIFACT_DIR = os.path.join(REPO_DIR, "models", "unet")
else:
    ARTIFACT_DIR = os.path.join(OUT_DIR, "unet-smoke")
    sh(f"{PY} -m floodvalue.model.unet.train --name unet_smoke --epochs 1 --max-chips 4 "
       f"--batch 2 --out-dir {shlex.quote(ARTIFACT_DIR)}", cwd=REPO_DIR, env=ENV)
print("артефакты:", ARTIFACT_DIR)

## 5. sha256 и выгрузка весов

`SHA256SUMS` рядом с весами; архив `unet_<MODE>_<commit>.zip` — в `MyDrive/floodvalue/`
(в Colab, `SAVE_TO_DRIVE = "1"`) или скачиванием через браузер. Дальше веса кладутся в
`models/unet/` рабочей копии и коммитятся через Git LFS (`models/README.md`), sha256 попадает
в `run_metadata.json`.

In [ ]:
import hashlib

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

# Only files written by this training run: models/unet/ of the clone may also hold older
# files from git (LFS pointers, a calibration of previous weights) that must not ship with them.
files = sorted(f for f in os.listdir(ARTIFACT_DIR)
               if os.path.isfile(os.path.join(ARTIFACT_DIR, f)) and f != "SHA256SUMS"
               and os.path.getmtime(os.path.join(ARTIFACT_DIR, f)) >= TRAIN_STARTED - 1)
stale = sorted(set(os.listdir(ARTIFACT_DIR)) - set(files) - {"SHA256SUMS"})
if stale:
    print("не из этого обучения, в архив не входят:", stale)
assert any(f.endswith(".pt") for f in files), f"no weights in {ARTIFACT_DIR}: {files}"
lines = [f"{sha256(os.path.join(ARTIFACT_DIR, f))}  {f}" for f in files]
with open(os.path.join(ARTIFACT_DIR, "SHA256SUMS"), "w") as f:
    f.write("\n".join(lines) + "\n")
print("\n".join(lines))

archive = shutil.make_archive(os.path.join(WORKDIR, f"unet_{MODE}_{COMMIT[:12]}"), "zip", ARTIFACT_DIR)
print("архив:", archive, os.path.getsize(archive), "байт")

if IN_COLAB:
    if SAVE_TO_DRIVE == "1":
        from google.colab import drive
        drive.mount("/content/drive")
        dest = "/content/drive/MyDrive/floodvalue"
        os.makedirs(dest, exist_ok=True)
        shutil.copy(archive, dest)
        print("скопировано в", dest)
    else:
        from google.colab import files as colab_files
        colab_files.download(archive)